# GameTheory 3f : Le Parcours Complet -- du jeu nommé au coût de la méta-action

[← GameTheory-3e](GameTheory-3e-Meta-Actions-Tarifees.ipynb) | [GameTheory-24-Chemin-Minimal →](GameTheory-24-Chemin-Minimal-Robinson-Goforth.ipynb) | [↑ README GameTheory](README.md)

**Versant d'intégration du chantier #12207.** Les versants précédents ont livré chacun une pièce : [GT-24](GameTheory-24-Chemin-Minimal-Robinson-Goforth.ipynb) construit et vérifie des chemins minimaux entre chambres, [GT-3b](GameTheory-3b-Chambres-et-Murs.ipynb) identifie les murs où vivent les égalités, [GT-3e](GameTheory-3e-Meta-Actions-Tarifees.ipynb) tarifé les swaps comme des actions que l'agent paie. Ce notebook referme la boucle : il réunit les trois pièces **dans un seul parcours**, celui que le chantier demande de bout en bout --

> un lecteur peut, dans un notebook exécuté : partir d'un **jeu nommé**, atteindre un autre jeu nommé par un **chemin qu'il n'a pas écrit lui-même**, **voir le mur** qu'il traverse à chaque pas, et **lire le coût** de la méta-action qui l'y a mené.

## Plan

1. **Le chemin que le lecteur n'écrit pas** : constructeur BFS et vérificateur indépendant (patron GT-24)
2. **Voir les murs** : chaque pas traverse un jeu à égalité codimension 1, dont on exhibe les deux faces
3. **Le coût des méta-actions** : tarifs par côté puis par niveau -- et un résultat inattendu, mesuré
4. **Le parcours complet, bout en bout** : le bloc intégral, puis sa re-vérification indépendante
5. **Exercices**

## La question

La théorie des jeux ordinaire décrit des agents *dans* des règles. La strate méta demande ce qui se passe quand **déplacer le jeu est une action payée** : quel trajet, quels murs franchis, à quel prix, et payé par qui. Sur l'univers fini de Robinson-Gofforth, chaque pièce de cette question est calculable -- ce notebook les calcule **ensemble**, sur des jeux portant un nom (Dilemme, Poule, Cerf), pour que le résultat se lise comme un récit et pas comme une coordonnée.

## 0. Le substrat, hérité de GT-3b / GT-3e / GT-24

Un jeu = **deux tables de rangs**, une par joueur, chacune un 4-uplet ordonnant strictement les quatre cases (haut-gauche, haut-droite, bas-gauche, bas-droite) -- rang 4 = meilleur. Les **swaps** échangent les cases portant deux rangs adjacents `k` et `k+1`, de part et d'autre : `R1, R2, R3` (côté Ligne) et `C1, C2, C3` (côté Colonne). Ce sont les six générateurs de l'espace ; chaque traversée d'un swap franchit un **mur** du monde de Bruns-Kimmich -- c'est l'objet de la section 2. Toute la mécanique est reprise telle quelle des trois versants, sans modification : ce notebook est un consommateur du substrat, pas une refonte.

In [1]:
# Substrat GT-3b / GT-3e / GT-24, repris tel quel -- pur stdlib
from itertools import product
from collections import Counter, deque
import heapq

def swap_valeurs_adjacentes(t, k):
    """Échange les cases portant les rangs k et k+1 (traversée de la facette k<->k+1)."""
    pk, pk1 = t.index(k), t.index(k + 1)
    l = list(t); l[pk], l[pk1] = l[pk1], l[pk]; return tuple(l)

def swap_jeu(jeu, cote, k):
    """Applique le swap de niveau k sur LA table du joueur désigné (méta-action unilatérale)."""
    row, col = jeu
    if cote == "ligne":
        return (swap_valeurs_adjacentes(row, k), col)
    return (row, swap_valeurs_adjacentes(col, k))

stricts = sorted({tuple(p) for p in product(range(1, 5), repeat=4) if len(set(p)) == 4})
chambres = [(r, c) for r in stricts for c in stricts]
print("Tables strictes par joueur :", len(stricts), "| chambres (jeux stricts) :", len(chambres))
print("Générateurs : R1 R2 R3 (Ligne) x C1 C2 C3 (Colonne)")

# Les cinq bornes canoniques (convention GT-21 / GT-3b / GT-24)
PD       = ((3, 1, 4, 2), (3, 4, 1, 2))   # T>R>P>S
POULE    = ((3, 2, 4, 1), (3, 4, 2, 1))   # T>R>S>P
CERF     = ((4, 1, 3, 2), (4, 3, 1, 2))   # R>T>P>S
IDENTITE = ((1, 2, 3, 4), (1, 2, 3, 4))
RENVERSE = ((4, 3, 2, 1), (4, 3, 2, 1))
for nom, j in [("PD (Dilemme)", PD), ("POULE (Poule)", POULE), ("CERF (Cerf)", CERF),
               ("IDENTITE", IDENTITE), ("RENVERSE", RENVERSE)]:
    ok = j in chambres
    print(f"  {nom:16s} Ligne {j[0]} | Colonne {j[1]} | chambre stricte : {ok}")

Tables strictes par joueur : 24 | chambres (jeux stricts) : 576
Générateurs : R1 R2 R3 (Ligne) x C1 C2 C3 (Colonne)
  PD (Dilemme)     Ligne (3, 1, 4, 2) | Colonne (3, 4, 1, 2) | chambre stricte : True
  POULE (Poule)    Ligne (3, 2, 4, 1) | Colonne (3, 4, 2, 1) | chambre stricte : True
  CERF (Cerf)      Ligne (4, 1, 3, 2) | Colonne (4, 3, 1, 2) | chambre stricte : True
  IDENTITE         Ligne (1, 2, 3, 4) | Colonne (1, 2, 3, 4) | chambre stricte : True
  RENVERSE         Ligne (4, 3, 2, 1) | Colonne (4, 3, 2, 1) | chambre stricte : True


### Lecture du substrat

Vingt-quatre tables par joueur, donc 24 x 24 = **576 chambres** -- l'univers de Robinson-Gofforth classique, déjà dérivé par GT-24 et non récité ici. Les cinq bornes canoniques y vivent : le Dilemme du Prisonnier (défection mutuelle stable), la Poule (le jeu du poulet), la Chasse au Cerf (coordination sur le meilleur monde commun), et les deux extrêmes structurants Identité et Renversement.

Deux précisions de vocabulaire pour la suite. Un **pas** relie deux chambres adjacentes : il modifie exactement une table, celle du joueur qui agit, par exactement un swap de niveau `k`. Et une **méta-action** (GT-3e) est ce pas *considéré comme payé* : réécrire une préférence déclarée coûte quelque chose, en échelons de rang -- c'est la section 3 qui fixe le barème.

## 1. Le chemin que le lecteur n'écrit pas

Le premier impératif du parcours : le chemin ne doit pas être écrit à la main. Si le lecteur choisissait lui-même ses swaps, le notebook ne montrerait rien -- il redirait ses propres intuitions. Le constructeur ci-dessous produit donc le chemin par **BFS avec remontée des parents**, exactement selon le patron de GT-24 : le graphe des 576 chambres est exploré depuis le départ, et la chaîne est reconstruite en remontant de l'arrivée.

Le second impératif est la **séparation constructeur / vérificateur** (loi II de la série) : celui qui produit le témoin n'est pas celui qui le juge. La cellule suivante héberge un vérificateur qui ne réutilise *rien* du constructeur -- il re-dérive les distances par sa propre recherche exhaustive.

In [2]:
# === Section 1.1 : construire_chemin -- le constructeur de témoin (patron GT-24) ===

def construire_chemin(depart, arrivee):
    """Produit une suite (jeu_avant, jeu_apres) de départ à arrivée par BFS + remontée des parents."""
    parent = {depart: None}
    q = deque([depart])
    while q:
        u = q.popleft()
        if u == arrivee:
            break
        for cote in ("ligne", "colonne"):
            for k in (1, 2, 3):
                v = swap_jeu(u, cote, k)
                if v not in parent:
                    parent[v] = u
                    q.append(v)
    if arrivee not in parent:
        return None
    chaine = []
    cur = arrivee
    while parent[cur] is not None:
        chaine.append((parent[cur], cur))
        cur = parent[cur]
    return chaine[::-1]

def nommer_pas(pred, cur):
    """Nomme le pas élémentaire entre deux jeux consécutifs (côté, niveaux échangés)."""
    for cote, ti in (("ligne", 0), ("colonne", 1)):
        if pred[ti] != cur[ti]:
            for k in (1, 2, 3):
                if swap_jeu(pred, cote, k) == cur:
                    nom_cote = "R" if cote == "ligne" else "C"
                    return f"{nom_cote}{k} (côté {cote}, niveaux {k} <-> {k + 1})"
    return "?"

chemin_pd_cerf = construire_chemin(PD, CERF)
print("Chemin construit : Dilemme du Prisonnier -> Chasse au Cerf")
print(f"  départ  PD   : Ligne {PD[0]} | Colonne {PD[1]}")
for pred, cur in chemin_pd_cerf:
    print(f"  -- {nommer_pas(pred, cur)} -->  Ligne {cur[0]} | Colonne {cur[1]}")
print(f"Longueur du chemin produit : {len(chemin_pd_cerf)} pas")

Chemin construit : Dilemme du Prisonnier -> Chasse au Cerf
  départ  PD   : Ligne (3, 1, 4, 2) | Colonne (3, 4, 1, 2)
  -- R3 (côté ligne, niveaux 3 <-> 4) -->  Ligne (4, 1, 3, 2) | Colonne (3, 4, 1, 2)
  -- C3 (côté colonne, niveaux 3 <-> 4) -->  Ligne (4, 1, 3, 2) | Colonne (4, 3, 1, 2)
Longueur du chemin produit : 2 pas


### Lecture : deux révisions du sommet

Le chemin construit a **2 pas** -- et chacun réécrit le **sommet** des préférences d'un joueur (niveaux 3 <-> 4). Ce n'est pas un choix du constructeur, c'est la géométrie : pour passer du Dilemme au Cerf, Ligne doit échanger ses deux meilleures cases (sa table `(3, 1, 4, 2)` devient `(4, 1, 3, 2)`) et Colonne doit faire de même. En langage des noms : dans le Dilemme, la défection est le sommet de chacun ; dans le Cerf, c'est la coordination. Aucun détour par les bas niveaux ne raccourcit ce trajet -- il faut toucher au sommet, deux fois.

On notera l'asymétrie apparente des tables du Cerf (`(4, 1, 3, 2)` et `(4, 3, 1, 2)`) : les deux joueurs y placent le même sommet mais ne classent pas pareil les deux pires cases -- le Cerf n'exige pas d'êtres identiques, seulement de préférer le monde commun.

In [3]:
# === Section 1.2 : verifier_chemin -- le vérificateur indépendant (patron GT-24) ===

def bfs_complet(depart):
    """Distances exhaustives depuis depart (la preuve de minimalité, recalculée par le vérificateur)."""
    dist = {depart: 0}
    q = deque([depart])
    while q:
        u = q.popleft()
        for cote in ("ligne", "colonne"):
            for k in (1, 2, 3):
                v = swap_jeu(u, cote, k)
                if v not in dist:
                    dist[v] = dist[u] + 1
                    q.append(v)
    return dist

def pas_elementaire_valide(pred, cur):
    """True ssi cur s'obtient de pred par exactement un swap élémentaire."""
    return cur in [swap_jeu(pred, cote, k) for cote in ("ligne", "colonne") for k in (1, 2, 3)]

def verifier_chemin(depart, arrivee, chaine):
    """Verdict indépendant : re-dérive TOUT, ne réutilise rien du constructeur."""
    if chaine is None or len(chaine) == 0:
        return "INVALIDE : chemin vide"
    if chaine[0][0] != depart or chaine[-1][1] != arrivee:
        return "INVALIDE : les extrémités ne sont pas celles annoncées"
    for pred, cur in chaine:
        if not pas_elementaire_valide(pred, cur):
            return f"INVALIDE : pas non élémentaire vers {cur}"
    d_reel = bfs_complet(depart)[arrivee]
    if len(chaine) != d_reel:
        return f"VALIDE mais NON MINIMAL : {len(chaine)} pas, distance réelle {d_reel}"
    return f"VALIDE + MINIMAL : {len(chaine)} pas = distance exhaustive d(G,H)"

print("1. Le chemin du constructeur (PD -> CERF) :")
print("  ", verifier_chemin(PD, CERF, chemin_pd_cerf))

voisin_cerf = swap_jeu(CERF, "ligne", 1)
chemin_detour = chemin_pd_cerf + [(CERF, voisin_cerf), (voisin_cerf, CERF)]
print("2. Le même chemin rallongé d'un aller-retour :")
print("  ", verifier_chemin(PD, CERF, chemin_detour))

print("3. Un 'chemin' en un pas vers un jeu non adjacent :")
print("  ", verifier_chemin(PD, POULE, [(PD, POULE)]))

1. Le chemin du constructeur (PD -> CERF) :
   VALIDE + MINIMAL : 2 pas = distance exhaustive d(G,H)
2. Le même chemin rallongé d'un aller-retour :
   VALIDE mais NON MINIMAL : 4 pas, distance réelle 2
3. Un 'chemin' en un pas vers un jeu non adjacent :
   INVALIDE : pas non élémentaire vers ((3, 2, 4, 1), (3, 4, 2, 1))


### Lecture : trois verdicts, une séparation

Le vérificateur rend trois verdicts distincts sur trois cas construits pour eux : `VALIDE + MINIMAL` pour le témoin du constructeur, `NON MINIMAL` pour le même chemin artificiellement rallongé d'un aller-retour, et `INVALIDE` pour un saut direct PD -> POULE (deux jeux que rien ne relie en un pas). La minimalité n'est pas une impression : elle est **re-dérivée par recherche exhaustive** depuis le départ, indépendamment de la façon dont le constructeur a trouvé son chemin.

C'est la loi II de la série, appliquée au parcours : le constructeur (BFS + parents) *produit* un témoin, le vérificateur (re-dérivation complète) le *juge*. À la section 4, ce couple sera étendu aux murs et aux coûts : le vérificateur final re-vérifiera tout ce que le parcours affiche.

## 2. Voir les murs

Entre deux chambres adjacentes -- avant et après un swap de niveau `k` -- il y a un **mur** : un jeu à rangs où les deux niveaux `k` et `k+1` sont tombés **ex æquo**. C'est la lecture Bruns-Kimmich reprise de GT-3b : le monde complet des jeux à rangs compte 5625 points, dont 576 chambres (aucune égalité) et des murs de codimension 1 (exactement une paire ex æquo). Traverser un swap, c'est littéralement *passer à travers* le mur qui sépare les deux chambres.

Deux opérations réciproques le formalisent : **fusionner** les niveaux `k` et `k+1` d'une chambre donne le mur qu'un pas de niveau `k` traverse ; **briser le tie** d'un mur redonne les deux chambres qu'il sépare. La propriété qui fait tenir la section : briser le tie du mur d'un pas doit redonner *exactement* les deux tables extrémités de ce pas. Elle est vérifiée globalement ci-dessous, sur les 24 tables et les 3 niveaux.

In [4]:
# === Section 2.1 : le mur d'un pas -- fusion de niveaux et brisure de tie (GT-3b) ===

def mur_du_niveau(t, k):
    """L'ordre faible (codim 1) où les niveaux k et k+1 de t sont ex æquo : le mur du pas de niveau k."""
    l = [x - 1 if x > k else x for x in t]      # les niveaux au-dessus descendent d'un cran
    l[t.index(k)] = k
    l[t.index(k + 1)] = k                       # les deux niveaux fusionnent en k
    return tuple(l)

def briser_le_tie(t):
    """Les deux chambres strictes que le mur t sépare (réciproque exacte de mur_du_niveau)."""
    v = min(x for x in t if t.count(x) > 1)     # la valeur ex æquo (mur simple : unique)
    i, j = [p for p in range(4) if t[p] == v]
    inc = [x + 1 if x > v else x for x in t]    # les niveaux au-dessus remontent d'un cran
    A = list(inc); A[i], A[j] = v, v + 1
    B = list(inc); B[i], B[j] = v + 1, v
    return tuple(A), tuple(B)

# Propriété clé, vérifiée GLOBALEMENT : briser le tie du mur d'un pas redonne ses extrémités
prop = all(sorted(briser_le_tie(mur_du_niveau(t, k))) == sorted([t, swap_valeurs_adjacentes(t, k)])
           for t in stricts for k in (1, 2, 3))
print("Propriété mur <-> paire de chambres, vérifiée sur 24 tables x 3 niveaux :", prop)

murs_par_joueur = {mur_du_niveau(t, k) for t in stricts for k in (1, 2, 3)}
print("Murs simples distincts par joueur :", len(murs_par_joueur), "(GT-3b mesurait 36)")

t, k = PD[0], 3
print(f"Exemple : table Ligne du Dilemme {t} | pas de niveau 3")
print(f"  mur          : {mur_du_niveau(t, 3)} (codim {4 - len(set(mur_du_niveau(t, 3)))})")
print(f"  faces du mur : {briser_le_tie(mur_du_niveau(t, 3))[0]}, {briser_le_tie(mur_du_niveau(t, 3))[1]}")
print(f"  extrémités   : {t} et {swap_valeurs_adjacentes(t, 3)}")

Propriété mur <-> paire de chambres, vérifiée sur 24 tables x 3 niveaux : True
Murs simples distincts par joueur : 36 (GT-3b mesurait 36)
Exemple : table Ligne du Dilemme (3, 1, 4, 2) | pas de niveau 3
  mur          : (3, 1, 3, 2) (codim 1)
  faces du mur : (3, 1, 4, 2), (4, 1, 3, 2)
  extrémités   : (3, 1, 4, 2) et (4, 1, 3, 2)


### Lecture : le mur a deux faces, et ce sont les extrémités du pas

La propriété est vérifiée sur les 72 combinaisons (24 tables x 3 niveaux) : **briser le tie du mur redonne exactement la chambre de départ et la chambre d'arrivée du pas**. Le mur n'est donc pas une métaphore décorative -- c'est l'objet géométrique *entre* les deux jeux, celui dont les deux faces sont les extrémités du pas. Sur l'exemple affiché : la table Ligne du Dilemme `(3, 1, 4, 2)` et sa voisine de niveau 3 `(4, 1, 3, 2)` -- la table du Cerf -- sont les deux faces du mur `(3, 1, 3, 2)`, où les cases haut-gauche et bas-gauche sont ex æquo au sommet.

Le comptage rejoint GT-3b : **36 murs simples par joueur** (24 chambres x 3 niveaux, chaque mur compté deux fois car chaque mur touche exactement 2 chambres). Chaque niveau de préférence a ses murs ; franchir un mur de niveau 3, c'est réécrire un sommet -- le lien avec le coût de la section suivante est direct.

In [5]:
# === Section 2.2 : les murs du chemin PD -> CERF, pas à pas ===

print("Le chemin PD -> CERF, lu mur par mur :")
for n, (pred, cur) in enumerate(chemin_pd_cerf, 1):
    cote, ti = ("ligne", 0) if pred[0] != cur[0] else ("colonne", 1)
    k = [kk for kk in (1, 2, 3) if swap_jeu(pred, cote, kk) == cur][0]
    mur = mur_du_niveau(pred[ti], k)
    faces = briser_le_tie(mur)
    confirmee = sorted(faces) == sorted([pred[ti], cur[ti]])
    print(f"  pas {n} : {nommer_pas(pred, cur)}")
    print(f"     table {cote} : {pred[ti]} -> {cur[ti]}")
    print(f"     mur traversé : {mur} (codim {4 - len(set(mur))}) | faces : {faces[0]}, {faces[1]}")
    print(f"     les faces sont les extrémités du pas : {confirmee}")

Le chemin PD -> CERF, lu mur par mur :
  pas 1 : R3 (côté ligne, niveaux 3 <-> 4)
     table ligne : (3, 1, 4, 2) -> (4, 1, 3, 2)
     mur traversé : (3, 1, 3, 2) (codim 1) | faces : (3, 1, 4, 2), (4, 1, 3, 2)
     les faces sont les extrémités du pas : True
  pas 2 : C3 (côté colonne, niveaux 3 <-> 4)
     table colonne : (3, 4, 1, 2) -> (4, 3, 1, 2)
     mur traversé : (3, 3, 1, 2) (codim 1) | faces : (3, 4, 1, 2), (4, 3, 1, 2)
     les faces sont les extrémités du pas : True


### Lecture : deux murs, un par joueur

Le chemin PD -> CERF traverse exactement **deux murs**, un par joueur, tous deux de niveau 3. Le mur de Ligne, `(3, 1, 3, 2)`, est le jeu où Ligne est *indifférent entre défection et coopération* (ses cases haut-gauche et bas-gauche ex æquo au sommet) tandis que la table de Colonne est encore celle du Dilemme. Le mur de Colonne, `(3, 3, 1, 2)`, est le miroir : Colonne indifférent au sommet, Ligne déjà converti au Cerf.

C'est le contenu concret du critère d'intégration : le chemin n'est plus une suite de coordonnées, c'est une suite de **situations intermédiaires lisibles** -- chacune est un jeu où un joueur exactement est suspendu entre ses deux anciens idéaux. La ligne `confirmee : True` de chaque pas est la garantie que le mur exhibé est bien *celui-là* et pas un voisin.

## 3. Le coût des méta-actions

Chaque pas du chemin est une **méta-action** au sens de GT-3e : un joueur réécrit ses préférences déclarées, et cette réécriture est payée en échelons de rang -- l'unité honnête d'un monde purement ordinal (payer 1, c'est renoncer à un échelon pour atteindre le niveau visé). Reste à fixer le **barème**. Deux lectures naturelles :

- **par côté** : Ligne paie `c_L` par swap, Colonne `c_C` -- le tarif différencié, par exemple la réécriture de la table Colonne (l'"institution") plus cher que celle de Ligne ;
- **par niveau** : réécrire le bas de sa liste coûte 1, le milieu 2, le **sommet 3** -- réviser ce qu'on préfère *entre tout* est la révision la plus profonde.

La question qui semble s'imposer : le chemin le plus court en nombre de pas est-il le moins cher ? L'intuition crie *non* -- un détour par des niveaux bon marché pourrait battre le trajet direct. La mesure, elle, répond autrement.

In [6]:
# === Section 3.1 : tarifs par côté -- le coût est géométrique ===

def dist_tables(t0):
    """Distances du graphe des 24 tables d'UN joueur (générateurs : les 3 swaps de niveau)."""
    d = {t0: 0}; q = deque([t0])
    while q:
        u = q.popleft()
        for k in (1, 2, 3):
            v = swap_valeurs_adjacentes(u, k)
            if v not in d: d[v] = d[u] + 1; q.append(v)
    return d

NOMS = [("PD", PD), ("POULE", POULE), ("CERF", CERF), ("IDENTITE", IDENTITE), ("RENVERSE", RENVERSE)]
c_L, c_C = 1, 3
print(f"Tarifs par côté : Ligne {c_L}, Colonne {c_C}")
print(f"{'paire':>22s} | d_L | d_C | longueur | coût | Ligne paie | Colonne paie")
print("-" * 78)
for (n1, j1), (n2, j2) in [(a, b) for i, a in enumerate(NOMS) for b in NOMS[i + 1:]]:
    dL = dist_tables(j1[0])[j2[0]]
    dC = dist_tables(j1[1])[j2[1]]
    print(f"{n1:>10s} -> {n2:<10s} | {dL}  | {dC}  |    {dL + dC}     |  {c_L*dL + c_C*dC:>2d}  |     {c_L*dL}      |      {c_C*dC}")

Tarifs par côté : Ligne 1, Colonne 3
                 paire | d_L | d_C | longueur | coût | Ligne paie | Colonne paie
------------------------------------------------------------------------------
        PD -> POULE      | 1  | 1  |    2     |   4  |     1      |      3
        PD -> CERF       | 1  | 1  |    2     |   4  |     1      |      3
        PD -> IDENTITE   | 3  | 4  |    7     |  15  |     3      |      12
        PD -> RENVERSE   | 3  | 2  |    5     |   9  |     3      |      6
     POULE -> CERF       | 2  | 2  |    4     |   8  |     2      |      6
     POULE -> IDENTITE   | 4  | 5  |    9     |  19  |     4      |      15
     POULE -> RENVERSE   | 2  | 1  |    3     |   5  |     2      |      3
      CERF -> IDENTITE   | 4  | 5  |    9     |  19  |     4      |      15
      CERF -> RENVERSE   | 2  | 1  |    3     |   5  |     2      |      3
  IDENTITE -> RENVERSE   | 6  | 6  |    12     |  24  |     6      |      18


### Lecture : un théorème, pas une coïncidence

Les colonnes `d_L` et `d_C` sont les distances **dans le graphe des 24 tables d'un seul joueur** -- chacune recalculée par BFS indépendant. Et le tableau montre la structure : **la longueur du chemin minimal est exactement `d_L + d_C`**, et son coût exactement `c_L·d_L + c_C·d_C`. Ce n'est pas un accident d'échantillon, c'est un théorème de trois lignes :

> chaque pas modifie exactement **une** table ; la table de Ligne doit passer de `PD[0]` à `CERF[0]`, ce qui exige au moins `d_L` pas côté Ligne ; symétriquement au moins `d_C` côté Colonne. Tout chemin a donc au moins `d_L + d_C` pas et coûte au moins `c_L·d_L + c_C·d_C` -- et le chemin BFS atteint exactement ces deux bornes.

La conséquence est le résultat central de cette section : **le coût du trajet minimal n'est pas négociable**. Aucun choix de chemin -- aucune ruse, aucun détour -- ne change la facture : qui paie quoi est fixé par la géométrie des deux tables, avant tout parcours. Pour aller du Dilemme au Cerf à ces tarifs, Ligne paie `c_L` et Colonne `c_C` -- quel que soit le chemin minimal emprunté.

In [7]:
# === Section 3.2 : tarifs par niveau -- la mesure répond à l'intuition ===

W = {1: 1, 2: 2, 3: 3}   # bas 1, milieu 2, sommet 3

def cout_chemin(chaine, w=W):
    total = 0
    for pred, cur in chaine:
        cote, ti = ("ligne", 0) if pred[0] != cur[0] else ("colonne", 1)
        k = [kk for kk in (1, 2, 3) if swap_jeu(pred, cote, kk) == cur][0]
        total += w[k]
    return total

def chemin_min_cout(depart, arrivee, w=W):
    """Dijkstra : le chemin de coût minimal sous les tarifs w (pas forcément le plus court)."""
    dist = {depart: 0}; parent = {depart: None}; pq = [(0, depart)]
    while pq:
        d, u = heapq.heappop(pq)
        if u == arrivee: break
        if d > dist.get(u, 10**9): continue
        for cote in ("ligne", "colonne"):
            for k in (1, 2, 3):
                v = swap_jeu(u, cote, k)
                if d + w[k] < dist.get(v, 10**9):
                    dist[v] = d + w[k]; parent[v] = u; heapq.heappush(pq, (d + w[k], v))
    chaine = []; cur = arrivee
    while parent[cur] is not None:
        chaine.append((parent[cur], cur)); cur = parent[cur]
    return chaine[::-1]

print(f"Tarifs par niveau : {W} | le plus court chemin est-il le moins cher ?")
ecarts = 0
for (n1, j1), (n2, j2) in [(a, b) for i, a in enumerate(NOMS) for b in NOMS[i + 1:]]:
    bfs = construire_chemin(j1, j2)
    dij = chemin_min_cout(j1, j2)
    cb, cd = cout_chemin(bfs), cout_chemin(dij)
    ecart = cb - cd
    ecarts += (ecart > 0)
    print(f"  {n1:>8s} -> {n2:<8s} : BFS {len(bfs)} pas à {cb} | Dijkstra {len(dij)} pas à {cd} | écart {ecart}")
print(f"Paires nommées où un détour battrait le plus court chemin : {ecarts} / 10")

import random
random.seed(0)
ech = [(random.choice(chambres), random.choice(chambres)) for _ in range(300)]
disc = sum(1 for a, b in ech if cout_chemin(chemin_min_cout(a, b)) < cout_chemin(construire_chemin(a, b)))
print(f"Échantillon de 300 paires quelconques de chambres : {disc} détour(s) gagnant(s)")

# La non-unicité du témoin : deux chemins distincts, la même facture
bfs_pi = construire_chemin(PD, IDENTITE)
dij_pi = chemin_min_cout(PD, IDENTITE)
print()
print("PD -> IDENTITE : le BFS et le Dijkstra produisent deux chemins distincts")
print("  BFS       :", [nommer_pas(p, c) for p, c in bfs_pi][:4], "... coût", cout_chemin(bfs_pi))
print("  Dijkstra  :", [nommer_pas(p, c) for p, c in dij_pi][:4], "... coût", cout_chemin(dij_pi))

Tarifs par niveau : {1: 1, 2: 2, 3: 3} | le plus court chemin est-il le moins cher ?
        PD -> POULE    : BFS 2 pas à 2 | Dijkstra 2 pas à 2 | écart 0
        PD -> CERF     : BFS 2 pas à 6 | Dijkstra 2 pas à 6 | écart 0
        PD -> IDENTITE : BFS 7 pas à 14 | Dijkstra 7 pas à 14 | écart 0
        PD -> RENVERSE : BFS 5 pas à 10 | Dijkstra 5 pas à 10 | écart 0
     POULE -> CERF     : BFS 4 pas à 8 | Dijkstra 4 pas à 8 | écart 0
     POULE -> IDENTITE : BFS 9 pas à 16 | Dijkstra 9 pas à 16 | écart 0
     POULE -> RENVERSE : BFS 3 pas à 8 | Dijkstra 3 pas à 8 | écart 0
      CERF -> IDENTITE : BFS 9 pas à 17 | Dijkstra 9 pas à 17 | écart 0
      CERF -> RENVERSE : BFS 3 pas à 4 | Dijkstra 3 pas à 4 | écart 0
  IDENTITE -> RENVERSE : BFS 12 pas à 20 | Dijkstra 12 pas à 20 | écart 0
Paires nommées où un détour battrait le plus court chemin : 0 / 10


Échantillon de 300 paires quelconques de chambres : 0 détour(s) gagnant(s)

PD -> IDENTITE : le BFS et le Dijkstra produisent deux chemins distincts
  BFS       : ['R2 (côté ligne, niveaux 2 <-> 3)', 'R1 (côté ligne, niveaux 1 <-> 2)', 'R3 (côté ligne, niveaux 3 <-> 4)', 'C2 (côté colonne, niveaux 2 <-> 3)'] ... coût 14
  Dijkstra  : ['R2 (côté ligne, niveaux 2 <-> 3)', 'R1 (côté ligne, niveaux 1 <-> 2)', 'C2 (côté colonne, niveaux 2 <-> 3)', 'C1 (côté colonne, niveaux 1 <-> 2)'] ... coût 14


### Lecture : le plus court est aussi le moins cher -- mesuré, pas prouvé

L'intuition du détour économique est **réfutée par la mesure** : sur les 10 paires nommées et sur un échantillon de 300 paires quelconques de chambres, **aucun** chemin de coût minimal n'est plus long que le plus court chemin. Sous ces tarifs, la longueur minimale et le coût minimal sont atteints *simultanément* -- le Dijkstra et le BFS rendent la même facture. Ce fait est **mesuré**, pas démontré : la section 3.1 le prouve pour les tarifs par côté (théorème des bornes `d_L`, `d_C`), mais pour les tarifs par niveau, aucun argument court ne clôt la question -- c'est dit tel quel, et l'exercice 2 la rouvre.

Le dernier affichage montre l'autre face de la structure : pour PD -> IDENTITE, BFS et Dijkstra produisent **deux chemins distincts de même coût**. Le témoin n'est pas unique -- plusieurs routes mènent au même prix -- mais la facture, elle, ne varie pas. Réuni au théorème de la section 3.1, le paysage économique du parcours tient en une phrase : **le prix du voyage est un invariant géométrique ; le chemin, lui, est un choix parmi des égalités**.

## 4. Le parcours complet, bout en bout

Les trois pièces sont prêtes : le constructeur de chemin (section 1), les murs (section 2), le barème (section 3). La fonction ci-dessous les assemble dans **un seul bloc** -- celui que le critère d'intégration du chantier demande : un jeu nommé, un chemin construit et non écrit à la main, chaque mur traversé exhibé avec ses deux faces, chaque méta-action affichée avec son coût et son payeur, et la facture finale par joueur.

Le barème est celui de la section 3.2 (bas 1, milieu 2, sommet 3), le même pour les deux joueurs -- la profondeur de la révision fait le prix, pas l'identité de qui révise.

In [8]:
# === Section 4.1 : parcours_complet -- le bloc du critère d'intégration ===

def parcours_complet(depart, arrivee, w, nom_dep, nom_arr):
    """Le parcours intégral : chemin construit + murs traversés + coûts, affichés d'un seul bloc."""
    chaine = construire_chemin(depart, arrivee)
    print(f"PARCOURS COMPLET : {nom_dep} -> {nom_arr}")
    print(f"  barème par niveau : {w} (le sommet coûte le plus cher)")
    print(f"  {nom_dep} : Ligne {depart[0]} | Colonne {depart[1]}")
    cumul = 0
    par_joueur = {"ligne": 0, "colonne": 0}
    for n, (pred, cur) in enumerate(chaine, 1):
        cote, ti = ("ligne", 0) if pred[0] != cur[0] else ("colonne", 1)
        k = [kk for kk in (1, 2, 3) if swap_jeu(pred, cote, kk) == cur][0]
        mur = mur_du_niveau(pred[ti], k)
        cout = w[k]
        cumul += cout
        par_joueur[cote] += cout
        print(f"  pas {n} : {nommer_pas(pred, cur)}")
        print(f"          jeu {pred[0]}|{pred[1]} -> {cur[0]}|{cur[1]}")
        print(f"          mur traversé : table {cote} {mur} (codim 1), faces {briser_le_tie(mur)[0]}, {briser_le_tie(mur)[1]}")
        print(f"          coût {cout} payé par {cote} | cumul {cumul}")
    print(f"  {nom_arr} : Ligne {arrivee[0]} | Colonne {arrivee[1]}")
    print(f"  ARRIVÉE : {len(chaine)} pas | facture totale {cumul} (Ligne {par_joueur['ligne']}, Colonne {par_joueur['colonne']})")
    return chaine

chemin_final = parcours_complet(PD, CERF, W, "Dilemme (PD)", "Chasse au Cerf (CERF)")

PARCOURS COMPLET : Dilemme (PD) -> Chasse au Cerf (CERF)
  barème par niveau : {1: 1, 2: 2, 3: 3} (le sommet coûte le plus cher)
  Dilemme (PD) : Ligne (3, 1, 4, 2) | Colonne (3, 4, 1, 2)
  pas 1 : R3 (côté ligne, niveaux 3 <-> 4)
          jeu (3, 1, 4, 2)|(3, 4, 1, 2) -> (4, 1, 3, 2)|(3, 4, 1, 2)
          mur traversé : table ligne (3, 1, 3, 2) (codim 1), faces (3, 1, 4, 2), (4, 1, 3, 2)
          coût 3 payé par ligne | cumul 3
  pas 2 : C3 (côté colonne, niveaux 3 <-> 4)
          jeu (4, 1, 3, 2)|(3, 4, 1, 2) -> (4, 1, 3, 2)|(4, 3, 1, 2)
          mur traversé : table colonne (3, 3, 1, 2) (codim 1), faces (3, 4, 1, 2), (4, 3, 1, 2)
          coût 3 payé par colonne | cumul 6
  Chasse au Cerf (CERF) : Ligne (4, 1, 3, 2) | Colonne (4, 3, 1, 2)
  ARRIVÉE : 2 pas | facture totale 6 (Ligne 3, Colonne 3)


### Lecture : le récit complet, lisible d'un bloc

Le bloc répond terme à terme au critère. Le jeu de départ porte un nom (Dilemme) et l'arrivée aussi (Chasse au Cerf). Le chemin est **construit** par BFS -- le lecteur ne l'a pas écrit. Chaque pas affiche **son mur** : la table à égalité exactement entre les deux jeux, avec ses deux faces, qui sont les extrémités du pas. Et chaque pas affiche **son coût** : deux révisions de sommet à 3 échelons chacune, payées séparément -- Ligne paie 3, Colonne paie 3, facture totale 6.

Le récit économique se lit maintenant mot à mot : *sortir du Dilemme vers le Cerf ne coûte aucun bas niveau -- il ne coûte que des sommets*. Personne n'a besoin de réviser ses pires cases ; chacun doit seulement admettre que sa meilleure case change de nature. C'est précisément le contenu de GT-3e sur ce couple (l'indifference exacte de la fuite solitaire), désormais visible comme géométrie tarifée : les deux murs franchis sont tous deux des murs de sommet.

In [9]:
# === Section 4.2 : verifier_parcours -- la re-vérification indépendante, étendue ===

def verifier_parcours(depart, arrivee, chaine, w, nom_dep, nom_arr):
    """Re-dérive TOUT ce que parcours_complet affiche : extrémités, pas, murs, coûts, minimalités."""
    if chaine is None or len(chaine) == 0 or chaine[0][0] != depart or chaine[-1][1] != arrivee:
        return "INVALIDE : les extrémités ne sont pas celles annoncées"
    cumul = 0
    for pred, cur in chaine:
        cote, ti = ("ligne", 0) if pred[0] != cur[0] else ("colonne", 1)
        k = [kk for kk in (1, 2, 3) if swap_jeu(pred, cote, kk) == cur]
        if not k:
            return f"INVALIDE : pas non élémentaire vers {cur}"
        k = k[0]
        if sorted(briser_le_tie(mur_du_niveau(pred[ti], k))) != sorted([pred[ti], cur[ti]]):
            return f"INVALIDE : le mur exhibé ne sépare pas {pred} et {cur}"
        cumul += w[k]
    d_long = bfs_complet(depart)[arrivee]
    if len(chaine) != d_long:
        return f"INVALIDE : {len(chaine)} pas, distance réelle {d_long}"
    opt = chemin_min_cout(depart, arrivee, w)
    d_cout = cout_chemin(opt, w)
    if cumul != d_cout:
        return f"VALIDE mais NON MINIMAL EN COÛT : {cumul} payés, optimum {d_cout}"
    return (f"TEMOIN CONSTRUIT ET VÉRIFIÉ INDÉPENDAMMENT : {nom_dep} -> {nom_arr} | "
            f"{len(chaine)} pas = distance exhaustive | facture {cumul} = optimum Dijkstra | "
            f"chaque mur confirmé par ses faces")

print(verifier_parcours(PD, CERF, chemin_final, W, "Dilemme (PD)", "Chasse au Cerf (CERF)"))

voisin_cerf = swap_jeu(CERF, "ligne", 1)
print()
print("Contre-épreuve -- le même parcours rallongé d'un aller-retour gratuit pour les yeux :")
print("  ", verifier_parcours(PD, CERF, chemin_final + [(CERF, voisin_cerf), (voisin_cerf, CERF)], W, "PD", "CERF"))

TEMOIN CONSTRUIT ET VÉRIFIÉ INDÉPENDAMMENT : Dilemme (PD) -> Chasse au Cerf (CERF) | 2 pas = distance exhaustive | facture 6 = optimum Dijkstra | chaque mur confirmé par ses faces

Contre-épreuve -- le même parcours rallongé d'un aller-retour gratuit pour les yeux :
   INVALIDE : 4 pas, distance réelle 2


### Lecture : le témoin construit et vérifié indépendamment

Le vérificateur étendu ne se contente pas des extrémités et de l'élémentarité : il re-dérive **chaque mur** (brisure de tie recalculée, faces comparées aux extrémités affichées), **chaque coût** (barème réappliqué pas à pas), la **minimalité en longueur** (BFS exhaustif) et la **minimalité en coût** (Dijkstra indépendant) -- et rend son verdict en une ligne. La contre-épreuve finale montre qu'un chemin rallongé d'un aller-retour, invisible aux yeux s'il était bien imprimé, est attrapé par la distance réelle.

La terminologie est celle du steer #12205 : un chemin produit par un constructeur et accepté par un vérificateur séparé est un **témoin construit et vérifié indépendamment** -- pas une preuve au sens formel, mais un objet dont chaque affirmation affichée a été re-dérivée par un code qui ne partage rien avec celui qui l'a produite. C'est le standard de la série depuis GT-24, étendu ici aux murs et aux coûts.

## 5. Exercices

Trois exercices prolongent le parcours. Le premier retourne la section 2 (du pas vers le mur) dans l'autre sens ; le deuxième rouvre la question laissée ouverte en 3.2 ; le troisième fait courir le parcours complet sur un autre couple nommé. Comme toujours dans la série : le notebook doit s'exécuter de bout en bout même exercices non complétés -- chaque cellule est un stub inoffensif, à remplacer.

In [10]:
# Exercice 1 -- le mur du pas, dans l'autre sens
# On donne deux chambres adjacentes quelconques (pred, cur). Retrouver le mur traversé :
# 1. identifier le côté modifié et le niveau k (le pas) ;
# 2. construire le mur par fusion de niveaux ;
# 3. vérifier que briser_le_tie(mur) redonne exactement {pred, cur} -- la propriété de la section 2.
# Test attendu sur pred_ex -> cur_ex : côté colonne, niveau 1, mur (2, 3, 1, 1).

def mur_du_pas(pred, cur):
    """Retourne (cote, k, mur) du pas pred -> cur, ou None si non adjacent."""
    pass  # TODO étudiant : identifier cote et k, puis mur_du_niveau
    return None

pred_ex = ((2, 1, 4, 3), (3, 4, 1, 2))
cur_ex = ((2, 1, 4, 3), (3, 4, 2, 1))
print("Exercice 1 à compléter : mur_du_pas", pred_ex, "->", cur_ex)

Exercice 1 à compléter : mur_du_pas ((2, 1, 4, 3), (3, 4, 1, 2)) -> ((2, 1, 4, 3), (3, 4, 2, 1))


In [11]:
# Exercice 2 -- chercher le contre-exemple au fait mesuré
# La section 3.2 a MESURÉ (10 paires nommées + 300 paires quelconques) que le chemin le plus court
# est aussi le moins cher sous W = {1: 1, 2: 2, 3: 3}. Est-ce robuste à un autre barème ?
# 1. choisir un autre tarif, p.ex. W2 = {1: 1, 2: 1, 3: 3} ou {1: 3, 2: 1, 3: 1} (sommet bon marché !) ;
# 2. re-faire le balayage : pour chaque paire nommée et un échantillon de 300 paires,
#    comparer cout_chemin(construire_chemin) et cout_chemin(chemin_min_cout) ;
# 3. rapporter : un barème où un détour gagne existe-t-il ?
# Indice : commencer par le barème sommet bon marché -- que devient l'écart moyen ?

W2 = {1: 1, 2: 2, 3: 3}  # TODO étudiant : essayer d'autres barèmes
print("Exercice 2 à compléter : balayage sous barème", W2)

Exercice 2 à compléter : balayage sous barème {1: 1, 2: 2, 3: 3}


In [12]:
# Exercice 3 -- capstone : POULE -> RENVERSE, le parcours complet
# Faire courir parcours_complet de la Poule au Renversement sous W = {1: 1, 2: 2, 3: 3},
# puis passer le résultat à verifier_parcours. Questions de lecture :
# 1. combien de pas, et répartis comment entre les deux joueurs ?
# 2. quels niveaux sont révisés -- le trajet touche-t-il le sommet de qui que ce soit ?
# 3. la facture par joueur : qui paie le plus, et pourquoi la géométrie l'impose-t-elle (cf 3.1) ?

print("Exercice 3 à compléter : parcours POULE -> RENVERSE + vérification")

Exercice 3 à compléter : parcours POULE -> RENVERSE + vérification


## Conclusion

Le critère d'intégration du chantier est tenu, bout en bout et sur un seul écran : **un jeu nommé** (le Dilemme), **un chemin non écrit à la main** (BFS + parents, vérifié), **des murs visibles** (chaque pas exhibe la table à égalité entre les deux jeux, ses deux faces confirmées), **des coûts lus** (deux révisions de sommet, 3 échelons chacune, payées séparément) -- et l'arrivée **porte un nom** (la Chasse au Cerf).

Deux résultats structurent le parcours au-delà de l'assemblage. Le **théorème des bornes** (3.1) : le coût du trajet minimal se décompose en `c_L·d_L + c_C·d_C` où `d_L`, `d_C` sont des distances de tables individuelles -- la facture est un invariant géométrique, hors de portée de tout marchandage sur le choix du chemin. Et le **fait mesuré** (3.2) : sous tarification par niveau, le plus court chemin est aussi le moins cher, sur toutes les paires testées -- l'intuition du détour économique ne survit pas à la mesure.

**Pour aller plus loin** : l'arbitrage *migrer ou rester* (à quel prix un agent accepte-t-il de payer ces 6 échelons ?) est le sujet de [GT-3e](GameTheory-3e-Meta-Actions-Tarifees.ipynb) ; la théorie des murs et des chambres, celui de [GT-3b](GameTheory-3b-Chambres-et-Murs.ipynb) ; la séparation constructeur / vérificateur, celle de [GT-24](GameTheory-24-Chemin-Minimal-Robinson-Goforth.ipynb).